In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [3]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.1)
  from scipy.sparse import csr_matrix, issparse


In [4]:
print("Connecting to ElasticSearch...")
es = Elasticsearch("http://localhost:9200")
index_name = "ir2025_documents" # Το ευρετήριο από τη Φάση 1

Connecting to ElasticSearch...


In [5]:
print("Loading Queries...")
df_queries = pd.read_csv("data/queries.csv")
df_queries['ID'] = df_queries['ID'].astype(str)

Loading Queries...


In [6]:
print("Loading Transformer Model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading Transformer Model...


In [7]:
N = 200 # from ElasticSearch
k = 50  # from FAISS
output_file = "results/phase3_results.txt"
run_name = "Hybrid_ES_FAISS"

In [8]:
if not os.path.exists("results"):
    os.makedirs("results")

print(f"Starting Hybrid Search for {len(df_queries)} queries...")

Starting Hybrid Search for 10 queries...


In [9]:
with open(output_file, 'w') as f:
    for index, row in df_queries.iterrows():
        q_id = row['ID']
        q_text = row['Text']
        
        # Retrieval ElasticSearch (Top N=200)
        res = es.search(
            index=index_name, 
            query={"match": {"text": q_text}}, 
            size=N
        )
        hits = res['hits']['hits']
        
        # if elasticsearch does not find anything continue
        if not hits:
            continue
            
        # collect IDs and texts
        doc_ids = [hit['_id'] for hit in hits]
        doc_texts = [hit['_source']['text'] for hit in hits]
        
        # Create embeddings with Transformers
        q_emb = model.encode([q_text], convert_to_numpy=True).astype('float32')
        doc_embs = model.encode(doc_texts, convert_to_numpy=True).astype('float32')
        
        # Cosine Similarity
        faiss.normalize_L2(q_emb)
        faiss.normalize_L2(doc_embs)
        
        # Dynamic FAISS index
        dim = doc_embs.shape[1]
        faiss_index = faiss.IndexFlatIP(dim)
        faiss_index.add(doc_embs)
        
        # Search and Re-Rank
        # top k=50
        search_k = min(k, len(doc_ids))
        distances, indices = faiss_index.search(q_emb, search_k)
        
        # Write in file
        for rank, faiss_idx in enumerate(indices[0]):
            real_doc_id = doc_ids[faiss_idx]
            score = distances[0][rank]
            
            f.write(f"{q_id} Q0 {real_doc_id} {rank+1} {score:.4f} {run_name}\n")

print(f"Hybrid Search Complete! Results saved to {output_file}")

Hybrid Search Complete! Results saved to results/phase3_results.txt
